In [2]:
!pip install -U bitsandbytes transformers accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [1]:
"""
run_inference.py
-----------------
Loads the QLoRA fine-tuned Mistral-7B adapter and generates tri-modal
cyber attack scenarios from held-out validation samples.

HOW TO RUN
----------
On Google Colab (new session):
    1. Upload this script, your training_data.jsonl, and your output/ folder
       (or mount Google Drive where output/ is saved)
    2. Run Cell 0 below to install packages, then run this script

On local machine:
    pip install torch transformers peft bitsandbytes datasets accelerate
    python run_inference.py

ARGUMENTS
---------
--adapter     Path to the saved LoRA adapter folder   (default: ./output)
--data        Path to training_data.jsonl             (default: ./training_data.jsonl)
--base        Base model name                         (default: mistralai/Mistral-7B-Instruct-v0.3)
--samples     How many val samples to generate for    (default: 3)
--compare     Also run the base model (no adapter) for side-by-side comparison
--out         Save results to this .txt file          (default: inference_results.txt)
--index       Use a specific record index from the val split (optional)
"""
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


# =========================
# CONFIG
# =========================
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
TEST_FILE  = "/content/test_inputs.jsonl"

# 👉 path where you uploaded adapter files
ADAPTER_PATH = "/content/"   # must contain adapter_model.safetensors + adapter_config.json

OUTPUT_FILE = "/content/generated_scenarios.txt"
MAX_NEW_TOKENS = 300


# =========================
# LOAD MODEL (4-bit)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("[+] Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[+] Loading adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("[+] Model ready\n")


# =========================
# LOAD TEST DATA
# =========================
with open(TEST_FILE, "r") as f:
    test_data = [json.loads(line) for line in f]

print(f"[+] Loaded {len(test_data)} test samples\n")


# =========================
# PROMPT BUILDER (FOR YOUR TEST DATA)
# =========================
def build_prompt(record):
    input_text = record.get("input", "")
    return f"[INST] Generate a tri-modal cyber attack scenario.\n\n{input_text} [/INST]\n"


# =========================
# GENERATION FUNCTION
# =========================
def generate_output(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# =========================
# RUN INFERENCE + SAVE OUTPUT
# =========================
results = []

with open(OUTPUT_FILE, "w") as f:

    f.write("TRI-MODAL CYBER ATTACK SCENARIO GENERATION\n")
    f.write("=" * 80 + "\n\n")

    for i, record in enumerate(test_data):
        print(f"[+] Generating sample {i+1}/{len(test_data)}")

        prompt = build_prompt(record)
        output = generate_output(prompt)

        # Save nicely formatted
        f.write(f"\n{'='*80}\n")
        f.write(f"SAMPLE {i+1}\n")
        f.write(f"{'-'*80}\n\n")

        f.write("INPUT:\n")
        f.write(record.get("input", "") + "\n\n")

        f.write("OUTPUT:\n")
        f.write(output + "\n\n")

        results.append(output)

print(f"\n[✓] All outputs saved to: {OUTPUT_FILE}")

[+] Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

[+] Generating sample 8/10


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[+] Generating sample 9/10


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[+] Generating sample 10/10

[✓] All outputs saved to: /content/generated_scenarios.txt


In [5]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


# =========================
# CONFIG
# =========================
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
TEST_FILE  = "/content/test_inputs.jsonl"
ADAPTER_PATH = "/content/"   # contains adapter_model.safetensors + adapter_config.json
OUTPUT_FILE = "/content/generated_scenarios_output.txt"

MAX_NEW_TOKENS = 300


# =========================
# LOAD MODEL (4-bit)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("[+] Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[+] Loading adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("[+] Model ready\n")


# =========================
# LOAD TEST DATA
# =========================
with open(TEST_FILE, "r") as f:
    test_data = [json.loads(line) for line in f]

print(f"[+] Loaded {len(test_data)} test samples\n")


# =========================
# PROMPT (STRICT FORMAT)
# =========================
def build_prompt(record):
    input_text = record.get("input", "")

    return f"""
[INST]
You MUST generate output in EXACTLY THREE SECTIONS.

If any section is missing, the answer is WRONG.

FORMAT:

[NATURAL LANGUAGE]
(3–5 paragraphs)

[MATHEMATICAL EXPRESSION]
G = (N, S, V, P, FS)
Define each clearly.

[ATTACK GRAPH]
Stage 1:
Precondition:
Postcondition:

Stage 2:
Precondition:
Postcondition:

Stage 3:
Precondition:
Postcondition:

STRICT RULE:
- DO NOT repeat instructions
- DO NOT explain format
- ONLY output the 3 sections

INPUT:
{input_text}
[/INST]
"""

# =========================
# GENERATION
# =========================
def generate_output(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt from output
    return full_output.replace(prompt, "").strip()


# =========================
# CLEAN OUTPUT (SAFETY)
# =========================
def clean_output(text):
    if "[NATURAL LANGUAGE]" not in text:
        text = "[NATURAL LANGUAGE]\n" + text

    if "[MATHEMATICAL EXPRESSION]" not in text:
        text += "\n\n[MATHEMATICAL EXPRESSION]\nG = (N, S, V, P, FS)"

    if "[ATTACK GRAPH]" not in text:
        text += "\n\n[ATTACK GRAPH]\nStage 1:\nPrecondition:\nPostcondition:"

    return text


# =========================
# RUN + SAVE OUTPUT
# =========================
with open(OUTPUT_FILE, "w") as f:

    f.write("TRI-MODAL CYBER ATTACK SCENARIO GENERATION\n")
    f.write("=" * 80 + "\n\n")

    for i, record in enumerate(test_data):
        print(f"[+] Generating sample {i+1}/{len(test_data)}")

        prompt = build_prompt(record)
        output = generate_output(prompt)
        output = clean_output(output)

        # Save nicely formatted
        f.write(f"\n{'='*80}\n")
        f.write(f"SAMPLE {i+1}\n")
        f.write(f"{'-'*80}\n\n")

        f.write("INPUT:\n")
        f.write(record.get("input", "") + "\n\n")

        f.write("OUTPUT:\n")
        f.write(output + "\n\n")

print(f"\n[✓] Outputs saved to: {OUTPUT_FILE}")

[+] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[+] Loading adapter...
[+] Model ready

[+] Loaded 10 test samples

[+] Generating sample 1/10
[+] Generating sample 2/10
[+] Generating sample 3/10
[+] Generating sample 4/10
[+] Generating sample 5/10
[+] Generating sample 6/10
[+] Generating sample 7/10
[+] Generating sample 8/10
[+] Generating sample 9/10
[+] Generating sample 10/10

[✓] Outputs saved to: /content/generated_scenarios_output.txt


In [6]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


# =========================
# CONFIG
# =========================
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
TEST_FILE  = "/content/test_inputs.jsonl"
ADAPTER_PATH = "/content/"   # contains adapter_model.safetensors + adapter_config.json
OUTPUT_FILE = "/content/generated_scenarios_output_final.txt"

MAX_NEW_TOKENS = 600   # 🔥 increased for full output


# =========================
# LOAD MODEL (4-bit)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("[+] Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("[+] Loading adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("[+] Model ready\n")


# =========================
# LOAD TEST DATA
# =========================
with open(TEST_FILE, "r") as f:
    test_data = [json.loads(line) for line in f]

print(f"[+] Loaded {len(test_data)} test samples\n")


# =========================
# 🔥 MATCH TRAINING PROMPT
# =========================
SYSTEM_PROMPT = """You are a senior cybersecurity red team expert specializing in cloud infrastructure attacks.
Your task is to generate detailed, technically accurate cyber attack scenarios in a strict tri-modal format.

CRITICAL FORMATTING RULES — follow these exactly:
1. Your response MUST start with [NATURAL LANGUAGE]
2. Do NOT use markdown
3. Must include exactly:
[NATURAL LANGUAGE]
[MATHEMATICAL EXPRESSION]
[ATTACK GRAPH]
4. Never skip a section
5. Use G = (N, S, V, P, FS)
6. Use numbered stages with -> arrows
"""


def build_prompt(record):
    input_text = record.get("input", "")

    return f"""[INST]
{SYSTEM_PROMPT}

INPUT:
{input_text}

Generate the tri-modal output now.
[/INST]
"""


# =========================
# GENERATION (DETERMINISTIC)
# =========================
def generate_output(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # 🔥 IMPORTANT (better structure)
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove prompt
    return full_output.replace(prompt, "").strip()


# =========================
# CLEAN OUTPUT (SAFETY NET)
# =========================
def clean_output(text):
    if "[NATURAL LANGUAGE]" not in text:
        text = "[NATURAL LANGUAGE]\n" + text

    if "[MATHEMATICAL EXPRESSION]" not in text:
        text += "\n\n[MATHEMATICAL EXPRESSION]\nG = (N, S, V, P, FS)"

    if "[ATTACK GRAPH]" not in text:
        text += "\n\n[ATTACK GRAPH]\nStage 1 -> Stage 2 -> Stage 3"

    return text


# =========================
# RUN + SAVE OUTPUT
# =========================
with open(OUTPUT_FILE, "w") as f:

    f.write("TRI-MODAL CYBER ATTACK SCENARIO GENERATION\n")
    f.write("=" * 80 + "\n\n")

    for i, record in enumerate(test_data):
        print(f"[+] Generating sample {i+1}/{len(test_data)}")

        prompt = build_prompt(record)
        output = generate_output(prompt)
        output = clean_output(output)

        f.write(f"\n{'='*80}\n")
        f.write(f"SAMPLE {i+1}\n")
        f.write(f"{'-'*80}\n\n")

        f.write("INPUT:\n")
        f.write(record.get("input", "") + "\n\n")

        f.write("OUTPUT:\n")
        f.write(output + "\n\n")

print(f"\n[✓] Outputs saved to: {OUTPUT_FILE}")

[+] Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[+] Loading adapter...
[+] Model ready

[+] Loaded 10 test samples

[+] Generating sample 1/10
[+] Generating sample 2/10
[+] Generating sample 3/10
[+] Generating sample 4/10
[+] Generating sample 5/10
[+] Generating sample 6/10
[+] Generating sample 7/10
[+] Generating sample 8/10
[+] Generating sample 9/10
[+] Generating sample 10/10

[✓] Outputs saved to: /content/generated_scenarios_output_final.txt
